In [ ]:
# ========================================================
# 08_train_family_classifier.ipynb
# Anomaly-family classifier trained on Autoencoder embeddings
# ========================================================

import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ========================================================
# Ustawienia
# ========================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_STATE = 42

print("=== TRENING KLASYFIKATORA TYPÓW ANOMALII NA EMBEDDINGACH AE ===")
print("Urządzenie:", DEVICE)

# ========================================================
# Autoencoder architecture consistent with 03_autoencoder.ipynb
# ========================================================

class ImprovedAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
        )

        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# ========================================================
# Wczytanie AE
# ========================================================

feature_columns = pd.read_csv("../data/processed/feature_schema.csv")["feature"].tolist()
input_dim = len(feature_columns)

ae = ImprovedAutoencoder(input_dim).to(DEVICE)
ae.load_state_dict(torch.load("../models/best_autoencoder_final.pth", map_location=DEVICE))
ae.eval()

print("Wczytano AE.")
print("Liczba cech:", input_dim)

# ========================================================
# Funkcja do embeddingów bottleneck 32D
# ========================================================

def get_embeddings(df, batch_size=4096):
    x = torch.tensor(df.values, dtype=torch.float32)
    embeddings = []

    ae.eval()
    with torch.no_grad():
        for start in range(0, len(x), batch_size):
            batch = x[start:start + batch_size].to(DEVICE)
            z = ae.encoder(batch)
            embeddings.append(z.cpu().numpy())

    return np.vstack(embeddings)

# ========================================================
# Dane anomalne do klasyfikacji
# ========================================================

scenarios = [
    "guloader",
    "scanning",
    "njrat",
    "kongtuke1",
    "kongtuke2",
    "remcos",
    "xloader",
    "xworm",
    "phantomstealer",
]

X_list = []
y_list = []

summary = []

for sc in scenarios:
    path = f"../data/processed/{sc}_features.csv"
    df = pd.read_csv(path)

    # upewniamy się, że kolumny są zgodne
    df = df.reindex(columns=feature_columns, fill_value=0)

    emb = get_embeddings(df)

    X_list.append(emb)
    y_list.extend([sc] * len(emb))

    summary.append({
        "scenario": sc,
        "flows": len(df),
        "embeddings": emb.shape[0],
        "embedding_dim": emb.shape[1],
    })

X = np.vstack(X_list)
y = np.array(y_list)

summary_df = pd.DataFrame(summary)

print("\n=== PODSUMOWANIE DANYCH ===")
print(summary_df.to_string(index=False))
print("X shape:", X.shape)
print("y shape:", y.shape)

# ========================================================
# Train/test split
# ========================================================

# Przy bardzo małych klasach stratify może się wywalić,
# dlatego robimy prosty split bez stratify.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    shuffle=True
)

print("\nTrain:", X_train.shape)
print("Test:", X_test.shape)

# ========================================================
# Random Forest
# ========================================================

clf = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced_subsample",
    n_jobs=-1
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)

print("\n=== WYNIKI KLASYFIKATORA ===")
print("Accuracy:", round(acc, 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

# ========================================================
# Zapis modelu
# ========================================================

os.makedirs("../models", exist_ok=True)
os.makedirs("../results", exist_ok=True)

joblib.dump(clf, "../models/family_classifier_rf.pkl")

pd.DataFrame({
    "true": y_test,
    "pred": y_pred
}).to_csv("../results/family_classifier_test_predictions.csv", index=False)

summary_df.to_csv("../results/family_classifier_training_summary.csv", index=False)

print("\nZapisano model: ../models/family_classifier_rf.pkl")
print("Zapisano predykcje: ../results/family_classifier_test_predictions.csv")
print("Zapisano summary: ../results/family_classifier_training_summary.csv")